# カーネル密度推定

## サンプルそれぞれの寄与度

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import random


random.seed(10)
np.random.seed(1)

sample_size=15

mean=3 #サンプルを取る正規分布の平均
sigma=2 #サンプルを取る正規分布の分散

# 正規分布の定義
def normal(x,mean,sigma):
    return 1/np.sqrt(2*np.pi*sigma**2)*np.exp(-((x-mean)/sigma)**2)

# サンプルの発生
y=list(np.random.normal(loc=mean,scale=sigma,size=int(sample_size)))
sample=sorted(y)

fig,ax=plt.subplots(nrows=2,figsize=(20,15))

# サンプルを取った正規分布の描画
ax1=ax[0]
x_kernel=np.linspace(np.min(y),np.max(y))
ax1.plot(x_kernel,[normal(x_kernel[i],mean,sigma) for i in range(len(x_kernel))],label='source',linewidth=5,color='cyan')

# ガウスカーネル関数の定義
def normal_kernel(x):
    return 1/np.sqrt(2*np.pi)*np.exp(-x**2/2)

# Scott's ruleによるバンド幅の設定（後述）
band_width=np.sqrt(np.var(sample,ddof=1)*((sample_size)**(-1/5))**2)

cmap=["#"+''.join([random.choice('0123456789ABCDEF') for j in range(6)]) for i in range(len(sample))]

# 各カーネル関数へ(x-x_i)/hを代入し、1/nhを掛けた関数の描画
output=[]
for i in range(len(sample)):
    y_kernel=1/(sample_size*band_width)*normal_kernel((x_kernel-sample[i])/band_width)
    output.append(y_kernel)
    ax1.plot(x_kernel,y_kernel,color=cmap[i])
    ax1.vlines(sample[i],0,0.1) #x_iの点の描画

ax1.legend(fontsize=30)

ax2=ax[1]
# それぞれのカーネル関数の寄与を描画
ax2.stackplot(x_kernel,output,colors=cmap)

# KDE関数の定義
# sampleはKDEの対象データ
def kde(x,sample,band_width,kernel):
    n=len(sample)
    return np.sum([1/(n*band_width)*kernel((x-sample[i])/band_width) for i in range(n)])
# KDEの結果を描画
x=np.linspace(np.min(sample),np.max(sample))
y=[kde(x[i],sample,band_width,normal_kernel) for i in range(len(x))]
ax2.plot(x,y,color='k',linewidth=10,label='kde',linestyle='dashed')

# サンプルを取った正規分布の描画
y=[normal(x[i],mean,sigma) for i in range(len(x))]
ax2.plot(x,y,color='cyan',linewidth=10,linestyle='dashed',label='source')

ax2.legend(fontsize=30)


## バンド幅の設定

In [ ]:
# 3種類のバンド幅のオプション
h=[0.05,0.3,1]

# サンプル数５
sample_size=5

y=list(np.random.normal(loc=mean,scale=sigma,size=int(sample_size)))
sample=sorted(y)

x_kernel=np.linspace(np.min(y)-1,np.max(y)+1,500)
cmap=["#"+''.join([random.choice('0123456789ABCDEF') for j in range(6)]) for i in range(len(sample))]

fig,ax=plt.subplots(nrows=3,figsize=(20,15))

# それぞれのバンド幅についてグラフを描く
for k in range(3):
    h_k=h[k]
    ax_k=ax[k]
    for i in range(len(sample)):
        y_kernel=[1/(sample_size*h_k)*normal_kernel((x_kernel[j]-sample[i])/h_k) for j in range(len(x_kernel))]
        ax_k.plot(x_kernel,y_kernel,color=cmap[i])
        ax_k.vlines(sample[i],0,np.max(y_kernel)*1.1)
    ax_k.set_title(f'bandwidth={h_k}',fontsize=20)
